In [1]:
import vectorbt as vbt
import pandas as pd
import numpy as np
from ggTrader.Signals import Signals
from utils.KrakenHistoricalData import KrakenHistoricalData



In [2]:

k = KrakenHistoricalData()

symbols = ["BTC"]
interval = "4h"
init_cash = 1000.0
transaction_fee = 0.004
start = pd.to_datetime("2025-01-01").tz_localize('UTC')
end = pd.to_datetime("2025-09-30").tz_localize('UTC')

df_multi = k.get_ohlcv_df(symbols, interval=interval)
df_multi.columns = df_multi.columns.droplevel(0)
df = df_multi.loc[start:end]

close = df['close']     # your close Series
# Indicator Params
params = {'sar_acceleration': 0.02,
          'sar_maximum': 0.2,
          'atr_multiplier': 3,
          'adx_threshold': 20,
          'adx_length': 14,
          'ce_high_length': 22,
          'ce_low_length': 22,
          'atr_length': 14,  # atr multiplier for chandelier exit
          }


In [3]:
s = Signals(**params)
signals = s.calc_signals(df.copy())
entries = signals['entry_signal']
exits = signals['exit_signal']



In [4]:
# Build target allocation:
# 1.0 = 100% of portfolio in asset
# 0.0 = 100% in cash
size = pd.Series(np.nan, index=close.index)
size[entries] = 1.0   # on buy signal: go all-in
size[exits] = 0.0     # on sell signal: go flat

pf = vbt.Portfolio.from_orders(
    close=close,
    size=size,
    size_type='targetpercent',  # interpret `size` as target weight
    direction='longonly',
    init_cash=init_cash,           # starting cash
    fees=transaction_fee,                 # 0.1% fee (optional)
    slippage=0.0005,            # optional
)



In [14]:
stats = pf.stats()

print(stats)
print(f"Sharpe ratio: {stats['Sharpe Ratio']}")

pf.plot(width=1600, height=1500).show()

Start                         2025-01-01 00:00:00+00:00
End                           2025-09-30 00:00:00+00:00
Period                                272 days 04:00:00
Start Value                                      1000.0
End Value                                    669.370437
Total Return [%]                             -33.062956
Benchmark Return [%]                          22.222137
Max Gross Exposure [%]                            100.0
Total Fees Paid                              172.192992
Max Drawdown [%]                              39.567495
Max Drawdown Duration                 253 days 08:00:00
Total Trades                                         27
Total Closed Trades                                  26
Total Open Trades                                     1
Open Trade PnL                                10.517568
Win Rate [%]                                  26.923077
Best Trade [%]                                14.986234
Worst Trade [%]                               -5